1. Project initialize and Overview

Import references and common method functions.

Cell 1 - Imports

In [ ]:
import os
import cv2
import time
import math
import json
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
from skimage.restoration import richardson_lucy, wiener

from ultralytics import YOLO

Cell 2 - Global Config

In [ ]:
# =========================================================
# Global configuration
# =========================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Original dataset structure
DATASET_ROOT = Path("gopro_deblur")
BLUR_DIR = DATASET_ROOT / "blur" / "images"
SHARP_DIR = DATASET_ROOT / "sharp" / "images"

# Extracted subset
SUBSET_ROOT = Path("subset_200")
SUBSET_BLUR_DIR = SUBSET_ROOT / "blur"
SUBSET_SHARP_DIR = SUBSET_ROOT / "sharp"

# Deblurred outputs
OUTPUT_ROOT = Path("outputs")
DEBLUR_RL_DIR = OUTPUT_ROOT / "deblurred" / "rl"
DEBLUR_WIENER_DIR = OUTPUT_ROOT / "deblurred" / "wiener"

# Logs and plots
LOGS_DIR = OUTPUT_ROOT / "logs"
PLOTS_DIR = OUTPUT_ROOT / "plots"
INFER_DIR = OUTPUT_ROOT / "inference"

# Candidate images for later manual annotation
CANDIDATE_ROOT = Path("detection_candidates")
CANDIDATE_IMAGES_DIR = CANDIDATE_ROOT / "images"
CANDIDATE_CSV = CANDIDATE_ROOT / "candidate_images.csv"

# Sampling
TARGET_TOTAL = 200
NUM_SEGMENTS = 10

# Deblur params
RL_ITERATIONS = 20
WIENER_BALANCE = 0.01
PSF_LENGTH = 15
PSF_ANGLE = 0

# YOLO
YOLO_MODEL_NAME = "yolov8n.pt"
CONF_THRES = 0.25
MAX_INFER_IMAGES = 200

# Create folders
for p in [
    SUBSET_BLUR_DIR, SUBSET_SHARP_DIR,
    DEBLUR_RL_DIR, DEBLUR_WIENER_DIR,
    LOGS_DIR, PLOTS_DIR, INFER_DIR,
    CANDIDATE_IMAGES_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("All directories are ready.")

Cell 3 — Utility Functions

In [ ]:
def list_image_files(folder):
    exts = {".png", ".jpg", ".jpeg", ".bmp"}
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts])

def load_image_rgb(path):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def save_image_rgb(path, img_rgb):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), img_bgr)

def to_float01(img):
    return img.astype(np.float32) / 255.0

def to_uint8(img):
    return np.clip(img * 255.0, 0, 255).astype(np.uint8)

def ensure_same_size(img1, img2):
    if img1.shape[:2] != img2.shape[:2]:
        img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    return img1, img2

2. Extract a 200-Pair Deblurring Subset

This part is responsible for constructing a subset of 200 pairs of blur/sharp from the original GoPro Deblur data. I did not upload the 1.9G data to GitHub. If you want to run this part of the code, you need to place the gopro_deblur files in the root directory. Here, I will generate a subset of 200 images and upload it to GitHub. 
This is the basic data pool for all subsequent experiments, used for:
Deblurring PSNR/SSIM YOLO inference analysis
Since the data does not have a "scene" subfolder, here we adopt: pairing by file name, overall segmentation, and interval sampling for each segment.

Cell 4 — Collect All Blur/Sharp Image Pairs

In [ ]:
blur_files = list_image_files(BLUR_DIR)
sharp_files = list_image_files(SHARP_DIR)

sharp_map = {p.name: p for p in sharp_files}

all_pairs = []
for b in blur_files:
    if b.name in sharp_map:
        all_pairs.append((b, sharp_map[b.name]))

print(f"Total matched pairs found: {len(all_pairs)}")

for i in range(min(5, len(all_pairs))):
    print(all_pairs[i][0].name, "<->", all_pairs[i][1].name)

Cell 5 — Split the Full Sequence into Segments

In [ ]:
def split_into_segments(items, num_segments=10):
    n = len(items)
    segments = []

    for i in range(num_segments):
        start = int(i * n / num_segments)
        end = int((i + 1) * n / num_segments)
        seg = items[start:end]
        if seg:
            segments.append(seg)

    return segments

segments = split_into_segments(all_pairs, NUM_SEGMENTS)

for i, seg in enumerate(segments):
    print(f"Segment {i+1}: {len(seg)} pairs, from {seg[0][0].name} to {seg[-1][0].name}")

Cell 6 — Define Interval Sampling Strategy

In [ ]:
def interval_sample(items, k):
    n = len(items)
    if k >= n:
        return items

    step = n / k
    indices = []

    for i in range(k):
        idx = int(i * step)
        if idx >= n:
            idx = n - 1
        indices.append(idx)

    indices = sorted(set(indices))

    if len(indices) < k:
        used = set(indices)
        for i in range(n):
            if i not in used:
                indices.append(i)
                used.add(i)
            if len(indices) == k:
                break

    indices = sorted(indices[:k])
    return [items[i] for i in indices]

Cell 7 — Allocate the 200 Samples Across Segments

In [ ]:
base_k = TARGET_TOTAL // len(segments)
remainder = TARGET_TOTAL % len(segments)

allocation = []
for i in range(len(segments)):
    k = base_k + (1 if i < remainder else 0)
    allocation.append(k)

print("Allocation per segment:", allocation)
print("Total allocation:", sum(allocation))

Cell 8 — Generate the Final 200-Pair Subset

In [ ]:
sampled_pairs = []

for seg, k in zip(segments, allocation):
    sampled = interval_sample(seg, k)
    sampled_pairs.extend(sampled)

print(f"Total sampled pairs: {len(sampled_pairs)}")
print("First 10 sampled filenames:")
for blur_path, sharp_path in sampled_pairs[:10]:
    print(blur_path.name)

Cell 9 — Copy the 200 Pairs to a New Subset Folder

In [ ]:
records = []

for blur_path, sharp_path in sampled_pairs:
    dst_blur = SUBSET_BLUR_DIR / blur_path.name
    dst_sharp = SUBSET_SHARP_DIR / sharp_path.name

    shutil.copy2(blur_path, dst_blur)
    shutil.copy2(sharp_path, dst_sharp)

    records.append({
        "filename": blur_path.name,
        "blur_path": str(dst_blur),
        "sharp_path": str(dst_sharp),
    })

subset_df = pd.DataFrame(records)
subset_df.to_csv(SUBSET_ROOT / "sampled_pairs.csv", index=False)

print(f"Copied {len(records)} pairs to {SUBSET_ROOT}")
subset_df.head()

Cell 10 — Rebuild Subset Pair List

In [ ]:
subset_pairs = []
sharp_subset_map = {p.name: p for p in list_image_files(SUBSET_SHARP_DIR)}

for b in list_image_files(SUBSET_BLUR_DIR):
    if b.name in sharp_subset_map:
        subset_pairs.append((b, sharp_subset_map[b.name]))

print(f"Subset matched pairs: {len(subset_pairs)}")

Cell 11 — Visual Sanity Check for the Extracted Subset

In [ ]:
num_show = 3
plt.figure(figsize=(12, 4 * num_show))

for i in range(num_show):
    blur_path, sharp_path = subset_pairs[i]
    blur_img = load_image_rgb(blur_path)
    sharp_img = load_image_rgb(sharp_path)

    plt.subplot(num_show, 2, 2 * i + 1)
    plt.imshow(blur_img)
    plt.title(f"Blur: {blur_path.name}")
    plt.axis("off")

    plt.subplot(num_show, 2, 2 * i + 2)
    plt.imshow(sharp_img)
    plt.title(f"Sharp: {sharp_path.name}")
    plt.axis("off")

plt.tight_layout()
plt.show()

3. Image Deblurring and Restoration Evaluation

This part requires implementing one or more defuzzification methods on 200 subsets and evaluating: 
restoration accuracy
computational cost
robustness

4. Object Detection Inference Analysis

Examine the impact of deblurring before and after on the inference results of the pre-trained object detection model.

5. Select and Prepare Images for Manual Annotation

Screen out a batch of candidate images from them, automatically tag them first, and then have humans confirm the final labeled set. 
First, conduct automatic pre-screening: 
Which pictures might contain the target category? 
Copy these images again to facilitate subsequent annotation with LabelImg.

6. Build YOLO Training Dataset from Annotated Images

This part is responsible for organizing the manually labeled images into the official YOLO dataset format and dividing them: 
train
val
test